In [1]:
import csv
import numpy as np
import os
import glob
import ase
from ase import io
import json
from glob import glob
from tqdm import tqdm
from pymatgen.io.vasp import Vasprun
from pymatgen.io.ase import AseAtomsAdaptor

import scipy
from scipy import interpolate
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d

In [2]:
#Settings
dos_length = 400
emin = -10
emax = 10

In [3]:
path = "/global/cfs/projectdirs/m3641/Shared/Materials_datasets/Addis_DOS_data"

In [4]:
xml_files = glob(f"{path}/**/*.xml", recursive=True)

In [5]:
xml_files.sort()

In [6]:
from pathlib import Path

In [7]:
files = [Path(x).parent.stem for x in xml_files]

In [8]:
len(files), len(set(files))

(488, 488)

In [9]:
count = 0
data_list = []

for xml_f in tqdm(xml_files):
    struct_name = Path(xml_f).parent.stem
    try:
        vasp_file = Vasprun(filename=xml_f)

        dos_temp = np.zeros((len(vasp_file.pdos), len(vasp_file.tdos.energies)))
        for i in range(0, len(vasp_file.pdos)):
            for orb in vasp_file.pdos[i]:
                for spin in vasp_file.pdos[i][orb]:
                    dos_temp[i,:] = dos_temp[i,:] +vasp_file.pdos[i][orb][spin]

        dos_interp = np.zeros((len(vasp_file.pdos), dos_length))
        for i in range(0, dos_interp.shape[0]):
            xfit=vasp_file.tdos.energies - vasp_file.tdos.efermi
            yfit=dos_temp[i,:]
            temp = interpolate.interp1d(xfit, yfit, kind="linear", bounds_error=False, fill_value=0)
            xnew = np.linspace(emin, emax, dos_length)
            dos_interp[i,:]=temp(xnew)

        ase_crystal = AseAtomsAdaptor.get_atoms(vasp_file.structures[-1])
        positions = ase_crystal.get_positions()
        cell = ase_crystal.get_cell()
        atomic_numbers = ase_crystal.get_atomic_numbers()
        data_dict = {
            'structure_id' : struct_name, 
            'positions' : positions.tolist(), 
            'cell' : cell.tolist(), 
            'atomic_numbers' : atomic_numbers.tolist(),  
            'y' : dos_interp.tolist()
        }

        data_list.append(data_dict)

        count += 1
    except Exception as e:
        print("error", e)

  0%|          | 0/488 [00:00<?, ?it/s]/global/homes/s/shuyijia/.conda/envs/spectrodiff-test/lib/python3.12/site-packages/pymatgen/io/vasp/outputs.py:1260: UserWarning: No POTCAR file with matching TITEL fields was found in

  if potcar := self.get_potcars(path):
/global/homes/s/shuyijia/.conda/envs/spectrodiff-test/lib/python3.12/site-packages/pymatgen/io/vasp/outputs.py:1278: UserWarning: No POTCAR file with matching TITEL fields was found in

  potcar = self.get_potcars(path)
/tmp/ipykernel_1407689/1321431924.py:7: UnconvergedVASPWarning: /global/cfs/projectdirs/m3641/Shared/Materials_datasets/Addis_DOS_data/cose2_Fe_adatom/vasprun.xml is an unconverged VASP run.
Electronic convergence reached: False.
Ionic convergence reached: True.
  vasp_file = Vasprun(filename=xml_f)
  0%|          | 1/488 [00:02<21:41,  2.67s/it]/tmp/ipykernel_1407689/1321431924.py:7: UnconvergedVASPWarning: /global/cfs/projectdirs/m3641/Shared/Materials_datasets/Addis_DOS_data/cose2_Fe_doped/vasprun.xml is an 

error no element found: line 1044, column 40


 64%|██████▎   | 310/488 [12:40<07:51,  2.65s/it]/tmp/ipykernel_1407689/1321431924.py:7: UnconvergedVASPWarning: /global/cfs/projectdirs/m3641/Shared/Materials_datasets/Addis_DOS_data/tase2-A_Vacancy-Anion-03/vasprun.xml is an unconverged VASP run.
Electronic convergence reached: False.
Ionic convergence reached: True.
  vasp_file = Vasprun(filename=xml_f)
 67%|██████▋   | 327/488 [13:25<07:09,  2.66s/it]/tmp/ipykernel_1407689/1321431924.py:7: UnconvergedVASPWarning: /global/cfs/projectdirs/m3641/Shared/Materials_datasets/Addis_DOS_data/tase2-B_Vacancy-Metal/vasprun.xml is an unconverged VASP run.
Electronic convergence reached: False.
Ionic convergence reached: True.
  vasp_file = Vasprun(filename=xml_f)
 79%|███████▉  | 386/488 [16:01<04:29,  2.64s/it]/tmp/ipykernel_1407689/1321431924.py:7: UnconvergedVASPWarning: /global/cfs/projectdirs/m3641/Shared/Materials_datasets/Addis_DOS_data/vs2_Adatom-Li/vasprun.xml is an unconverged VASP run.
Electronic convergence reached: False.
Ionic co

In [10]:
len(data_list)

487

In [11]:
with open('data.json', 'w') as f:
    json.dump(data_list , f)

In [19]:
keys = []

for x in xml_files:
    t = Path(x).parent.stem
    if t.startswith("tite2"):
        keys.append(t)

In [20]:
keys

['tite2-A_Defect-Free',
 'tite2-A_Vacancy-Anion-03',
 'tite2-B_Adatom-K',
 'tite2-B_Adatom-Li',
 'tite2-B_Adatom-Na',
 'tite2-B_Anti-Anion',
 'tite2-B_Nb_doped',
 'tite2-B_V_doped',
 'tite2-B_Vacancy-Anion-03',
 'tite2-B_Vacancy-Anion-09A',
 'tite2-B_Vacancy-Anion-09B',
 'tite2-B_Vacancy-Anion-12A',
 'tite2-B_Vacancy-Metal']

In [25]:
from collections import Counter

In [26]:
for each in data_list:
    if each['structure_id'] in keys:
        if len(each['atomic_numbers']) == 48:
            c = Counter(each['atomic_numbers'])
            print(each['structure_id'], c)

tite2-A_Defect-Free Counter({52: 32, 22: 16})
tite2-B_Anti-Anion Counter({52: 33, 22: 15})
tite2-B_Nb_doped Counter({52: 32, 22: 15, 41: 1})
tite2-B_V_doped Counter({52: 32, 22: 15, 23: 1})


In [29]:
for each in data_list:
    if each['structure_id'] == "tite2-A_Defect-Free":
        y = np.array(each['y'])

In [30]:
y.shape

(48, 400)

In [37]:
z = np.sum(y, axis=0)

In [38]:
z.shape

(400,)

In [39]:
z

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
      

In [40]:
len(data_list)

487

In [42]:
for x in data_list:
    atoms = ase.Atoms(
        numbers=x['atomic_numbers'],
        positions=x['positions'],
        cell=x['cell'],
        pbc=True
    )

    ase.io.write(f"data/2d_dos_cif/{x['structure_id']}.cif", atoms)